In [0]:
# Create gold, silver and bronze, run only once


CATALOG = "final_project_databi_group8"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`bronze`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`silver`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`gold`")

print(f"✓ {CATALOG}.bronze ready")
print(f"✓ {CATALOG}.silver ready")
print(f"✓ {CATALOG}.gold ready")

In [0]:
# query to check data in bronze 


CATALOG = "final_project_databi_group8"

#  Chicago Bronze 
print("=== CHICAGO BRONZE ===")
display(spark.sql(f"""
    SELECT
        city_source,
        LEFT(_batch_id, 8)          AS batch_short,
        _extract_date,
        _run_id,
        _schema_version,
        COUNT(*)                    AS row_count
    FROM {CATALOG}.bronze.chicago_raw
    GROUP BY city_source, _batch_id, _extract_date, _run_id, _schema_version
    ORDER BY _extract_date DESC
"""))

#  Dallas Bronze
print("=== DALLAS BRONZE ===")
display(spark.sql(f"""
    SELECT
        city_source,
        LEFT(_batch_id, 8)          AS batch_short,
        _extract_date,
        _run_id,
        _schema_version,
        COUNT(*)                    AS row_count
    FROM {CATALOG}.bronze.dallas_raw
    GROUP BY city_source, _batch_id, _extract_date, _run_id, _schema_version
    ORDER BY _extract_date DESC
"""))

#  Quick sanity check
chi_count = spark.table(f"{CATALOG}.bronze.chicago_raw").count()
dal_count = spark.table(f"{CATALOG}.bronze.dallas_raw").count()

print(f"\n Chicago Bronze : {chi_count:,} rows ")
print(f"\n Dallas Bronze  : {dal_count:,} rows ")

In [0]:
CATALOG = "final_project_databi_group8"

# ── Chicago Silver Clean ─────────────────────────────────────
print("=== CHICAGO SILVER CLEAN ===")
display(spark.sql(f"""
    SELECT
        city_source,
        COUNT(*)                                            AS total_inspections,
        ROUND(AVG(inspection_score), 2)                     AS avg_score,
        ROUND(AVG(violation_count), 2)                      AS avg_violations,
        SUM(CASE WHEN inspection_result = 'Pass'               THEN 1 ELSE 0 END) AS pass_count,
        SUM(CASE WHEN inspection_result = 'Fail'               THEN 1 ELSE 0 END) AS fail_count,
        SUM(CASE WHEN inspection_result = 'Pass w/ Conditions' THEN 1 ELSE 0 END) AS pass_cond,
        SUM(CASE WHEN risk_category = 'High'    THEN 1 ELSE 0 END) AS high_risk,
        SUM(CASE WHEN risk_category = 'Medium'  THEN 1 ELSE 0 END) AS medium_risk,
        SUM(CASE WHEN risk_category = 'Low'     THEN 1 ELSE 0 END) AS low_risk,
        MIN(inspection_date)                                AS earliest_date,
        MAX(inspection_date)                                AS latest_date
    FROM {CATALOG}.silver.chicago_clean
    GROUP BY city_source
"""))

# ── Chicago Quarantine ───────────────────────────────────────
print("=== CHICAGO QUARANTINE ===")
display(spark.sql(f"""
    SELECT
        _dqx_failed_checks,
        COUNT(*) AS row_count
    FROM {CATALOG}.silver.chicago_quarantine
    GROUP BY _dqx_failed_checks
    ORDER BY row_count DESC
    LIMIT 10
"""))

# ── Dallas Silver Clean ──────────────────────────────────────
print("=== DALLAS SILVER CLEAN ===")
display(spark.sql(f"""
    SELECT
        city_source,
        COUNT(*)                                            AS total_inspections,
        ROUND(AVG(inspection_score), 2)                     AS avg_score,
        ROUND(AVG(violation_count), 2)                      AS avg_violations,
        MIN(inspection_score)                               AS min_score,
        MAX(inspection_score)                               AS max_score,
        SUM(CASE WHEN inspection_type = 'Routine'   THEN 1 ELSE 0 END) AS routine,
        SUM(CASE WHEN inspection_type = 'Follow-up' THEN 1 ELSE 0 END) AS followup,
        SUM(CASE WHEN inspection_type = 'Complaint' THEN 1 ELSE 0 END) AS complaint,
        MIN(inspection_date)                                AS earliest_date,
        MAX(inspection_date)                                AS latest_date
    FROM {CATALOG}.silver.dallas_clean
    GROUP BY city_source
"""))

# ── Dallas Quarantine ────────────────────────────────────────
print("=== DALLAS QUARANTINE ===")
display(spark.sql(f"""
    SELECT
        _dqx_failed_checks,
        COUNT(*) AS row_count
    FROM {CATALOG}.silver.dallas_quarantine
    GROUP BY _dqx_failed_checks
    ORDER BY row_count DESC
    LIMIT 10
"""))

# ── Violations ───────────────────────────────────────────────
print("=== VIOLATIONS ===")
display(spark.sql(f"""
    SELECT
        city_source,
        COUNT(*)                        AS total_violation_rows,
        COUNT(DISTINCT violation_code)  AS unique_codes,
        COUNT(DISTINCT inspection_id)   AS inspections_with_violations
    FROM {CATALOG}.silver.chicago_violations
    GROUP BY city_source
"""))

display(spark.sql(f"""
    SELECT
        city_source,
        COUNT(*)                        AS total_violation_rows,
        COUNT(DISTINCT violation_code)  AS unique_codes,
        COUNT(DISTINCT inspection_id)   AS inspections_with_violations
    FROM {CATALOG}.silver.dallas_violations
    GROUP BY city_source
"""))

# ── Row count summary ────────────────────────────────────────
chi_clean  = spark.table(f"{CATALOG}.silver.chicago_clean").count()
dal_clean  = spark.table(f"{CATALOG}.silver.dallas_clean").count()
chi_quar   = spark.table(f"{CATALOG}.silver.chicago_quarantine").count()
dal_quar   = spark.table(f"{CATALOG}.silver.dallas_quarantine").count()
chi_viols  = spark.table(f"{CATALOG}.silver.chicago_violations").count()
dal_viols  = spark.table(f"{CATALOG}.silver.dallas_violations").count()

print(f"\n{'='*50}")
print(f"SILVER SUMMARY")
print(f"{'='*50}")
print(f"Chicago clean      : {chi_clean:,}  (expected ~253,000)")
print(f"Chicago quarantine : {chi_quar:,}   (expected ~86,000+)")
print(f"Chicago violations : {chi_viols:,}")
print(f"Dallas clean       : {dal_clean:,}   (expected ~54,000)")
print(f"Dallas quarantine  : {dal_quar:,}    (expected ~25,000)")
print(f"Dallas violations  : {dal_viols:,}")
print(f"{'='*50}")
print(f"Total clean rows   : {chi_clean + dal_clean:,}")